# Raw data profiling

Run all cells from top to bottom. Start with the inventory and review queue. Expand individual column details to see full frequencies, numeric buckets and date coverage.

The six CSVs are read as supplied. Blank fields are represented as missing and whitespace-only fields are counted separately. Parsed dates, numeric conversions and normalized text are diagnostic views. No rows are removed and no cleaned files are written. The README is a reference for the checks.

In [1]:
from pathlib import Path
from html import escape

import pandas as pd
from IPython.display import HTML, display

ROOT = next(path for path in (Path.cwd(), Path.cwd().parent) if (path / "data").is_dir())
DATA = ROOT / "data"
SOURCE = DATA / "raw"
EXTRACT_DATE = pd.Timestamp("2026-11-05", tz="UTC")
Q3_DATES = pd.date_range("2026-07-01", "2026-09-30", tz="UTC")
CHANNELS = ("Meta", "Google Ads", "TikTok")

# Keys describe the intended table grain. Subscription start date is a candidate key.
SCHEMA = {
    "users": {
        "key": ["user_id"], "dates": ["signup_ts", "kyc_verified_ts"], "numbers": [],
    },
    "transactions": {
        "key": ["txn_id"], "dates": ["txn_ts"], "numbers": ["amount_egp"],
    },
    "loans": {
        "key": ["loan_id"], "dates": ["disbursed_date", "due_date", "last_repayment_date"],
        "numbers": ["principal_egp", "fee_egp", "amount_repaid_egp"],
    },
    "plus_subscriptions": {
        "key": ["user_id", "plus_start_date"], "dates": ["plus_start_date", "plus_end_date"],
        "numbers": ["monthly_fee_egp"],
    },
    "cvm_campaigns": {
        "key": ["user_id", "campaign_id"], "dates": ["sent_ts"], "numbers": [],
    },
    "marketing_spend": {
        "key": ["date", "channel"], "dates": ["date"],
        "numbers": ["spend_egp", "impressions", "clicks", "platform_reported_installs"],
    },
}

tables = {}
inventory = []
for table, spec in SCHEMA.items():
    matches = sorted(SOURCE.glob(f"*{table}.csv"))
    if len(matches) != 1:
        raise ValueError(f"Expected one CSV for {table} in {SOURCE}, found {len(matches)}")
    frame = pd.read_csv(matches[0], dtype="string", keep_default_na=False, na_values=[""])
    tables[table] = frame
    inventory.append({
        "table": table, "source": matches[0].name, "rows": len(frame),
        "columns": len(frame.columns), "key": " + ".join(spec["key"]),
        "exact_duplicate_rows": int(frame.duplicated().sum()),
    })

inventory = pd.DataFrame(inventory).set_index("table")
display(inventory)

,source,rows,columns,key,exact_duplicate_rows
table,,,,,
users,02_Data & Insights - Analyst Case - users.csv,6230,7,user_id,0
transactions,03_Data & Insights - Analyst Case - transactio...,42378,6,txn_id,577
loans,04_Data & Insights - Analyst Case - loans.csv,1310,8,loan_id,0
plus_subscriptions,05_Data & Insights - Analyst Case - plus_subsc...,111,4,user_id + plus_start_date,0
cvm_campaigns,06_Data & Insights - Analyst Case - cvm_campai...,697,4,user_id + campaign_id,0
marketing_spend,07_Data & Insights - Analyst Case - marketing_...,276,6,date + channel,0


## Reusable profiling and checks

In [2]:
def role_of(column, spec):
    if column in spec["dates"]:
        return "date"
    if column in spec["numbers"]:
        return "numeric"
    if column.endswith("_id"):
        return "ID"
    return "text"


def has_value(series):
    return series.notna() & series.str.strip().ne("").fillna(False)


def dates(series):
    return pd.to_datetime(series, format="mixed", errors="coerce", utc=True)


def numbers(series):
    return pd.to_numeric(series, errors="coerce")


def column_summary(frame, spec):
    rows = []
    for column, series in frame.items():
        counts = series.value_counts()
        rows.append({
            "column": column, "role": role_of(column, spec), "rows": len(series),
            "missing": int(series.isna().sum()),
            "missing_pct": round(100 * series.isna().mean(), 2),
            "whitespace_only": int((series.notna() & ~has_value(series)).sum()),
            "distinct_non_null": len(counts),
            "distinct_values_repeated": int(counts.gt(1).sum()),
            "extra_occurrences": int((counts - 1).sum()),
            "extra_occurrences_pct": round(100 * (counts - 1).sum() / len(series), 2) if len(series) else None,
        })
    return pd.DataFrame(rows).set_index("column")


def frequency(series):
    return series.value_counts(dropna=False).rename("rows").to_frame()


def column_details(series, role):
    if role == "numeric":
        parsed = numbers(series)
        valid = parsed[~parsed.isin([float("inf"), float("-inf")])].dropna()
        stats = valid.describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]).to_frame("value")
        diagnostics = pd.Series({
            "parse_failures": (has_value(series) & parsed.isna()).sum(),
            "infinite": parsed.isin([float("inf"), float("-inf")]).sum(),
            "zero": valid.eq(0).sum(), "negative": valid.lt(0).sum(),
        }).to_frame("rows")
        buckets = pd.cut(valid.astype(float), bins=10).value_counts(sort=False) if len(valid) else valid
        extremes = pd.concat([
            valid.nsmallest(5).rename("value").to_frame().assign(tail="lowest"),
            valid.nlargest(5).rename("value").to_frame().assign(tail="highest"),
        ]).rename_axis("row_index")
        return [("Statistics", stats), ("Diagnostics", diagnostics),
                ("Equal-width buckets", buckets.rename("rows").to_frame()), ("Extremes", extremes)]

    if role == "date":
        parsed = dates(series)
        diagnostics = pd.Series({
            "earliest": parsed.min(), "latest": parsed.max(),
            "parse_failures": int((has_value(series) & parsed.isna()).sum()),
            "after_extract_day": int(parsed.ge(EXTRACT_DATE + pd.Timedelta(days=1)).sum()),
        }).to_frame("value")
        coverage = parsed.dropna().dt.strftime("%Y-%m-%d")
        return [("Date diagnostics", diagnostics), ("Daily coverage", coverage.value_counts().sort_index().to_frame("rows"))]

    if role == "ID":
        observed = series[has_value(series)]
        if len(observed) and observed.str.fullmatch(r"[0-9]+").all():
            identifiers = sorted(set(map(int, observed)))
            gaps = pd.DataFrame([
                {"after_id": left, "before_id": right, "missing_between": right - left - 1}
                for left, right in zip(identifiers, identifiers[1:]) if right - left > 1
            ], columns=["after_id", "before_id", "missing_between"])
            stats = pd.Series({
                "min_id": identifiers[0], "max_id": identifiers[-1],
                "distinct_integer_ids": len(identifiers), "gaps": len(gaps),
                "integers_skipped": int(gaps.missing_between.sum()),
            }).to_frame("value")
            return [("Observed integer continuity", stats), ("Gap intervals (first 20)", gaps.head(20))]

    normalized = series.str.strip().str.casefold()
    return [("Raw value counts", frequency(series)),
            ("Trimmed and case-folded counts", frequency(normalized))]


def display_details(label, sections):
    body = "".join(
        f"<h4>{escape(title)}</h4><div style='max-height:330px;overflow:auto'>"
        f"{frame.to_html(max_rows=None, escape=True, na_rep='&lt;missing&gt;')}</div>"
        for title, frame in sections
    )
    display(HTML(f"<details><summary>{escape(label)}</summary>{body}</details>"))

In [3]:
QA_NAMES = {"QA Lucky", "Test User", "test qa", "test test", "تجربه تطبيق", "تجربة"}
KNOWN_SOURCES = {
    "meta", "facebook ads", "facebook", "fb", "instagram", "ig",
    "google", "google ads", "adwords", "gads", "tiktok", "tt",
    "referral", "ref", "invite", "organic", "(direct)",
}
quality_rows = []
examples = {}


def record(table, check, matched, interpretation):
    # Matches are audit evidence. Some checks count expected or optional patterns.
    frame = tables[table]
    matched = matched.fillna(False)
    count = int(matched.sum())
    quality_rows.append({
        "table": table, "check": check, "rows": count,
        "pct_of_table": round(100 * count / len(frame), 2) if len(frame) else None,
        "interpretation": interpretation,
    })
    if count:
        examples[(table, check)] = frame.loc[matched].head(5)


raw_users = tables["users"]
known_ids = raw_users.user_id.dropna()
qa_names_normalized = {name.casefold() for name in QA_NAMES}
qa_mask = raw_users.full_name.str.strip().str.casefold().isin(qa_names_normalized)
qa_ids = raw_users.loc[qa_mask, "user_id"].dropna()
signups = dates(raw_users.dropna(subset=["user_id"]).drop_duplicates("user_id").set_index("user_id").signup_ts)

for table, frame in tables.items():
    spec = SCHEMA[table]
    key = spec["key"]
    key_present = frame[key].apply(has_value).all(axis=1)
    record(table, "Exact duplicate copies beyond first", frame.duplicated(), "Candidate for exact deduplication")
    record(table, "Missing key component", ~key_present, "Review intended grain")
    record(table, "Repeated complete key rows", key_present & frame.duplicated(key, keep=False), "Includes original and repeated rows")
    unique_rows = frame.drop_duplicates()
    conflict = unique_rows.duplicated(key, keep=False) & key_present.loc[unique_rows.index]
    record(table, "Distinct rows sharing key", conflict.reindex(frame.index, fill_value=False), "Inspect conflicting fields before choosing a rule")
    for column in spec["dates"]:
        parsed = dates(frame[column])
        record(table, f"{column}: date parse failure", has_value(frame[column]) & parsed.isna(), "Inspect source")
        if column != "due_date":
            record(table, f"{column}: after extract day", parsed.ge(EXTRACT_DATE + pd.Timedelta(days=1)), "Review timestamp")
    for column in spec["numbers"]:
        parsed = numbers(frame[column])
        record(table, f"{column}: numeric parse failure", has_value(frame[column]) & parsed.isna(), "Inspect source")
        record(table, f"{column}: negative", parsed.lt(0), "Review against field definition")
    if table != "users" and "user_id" in frame:
        record(table, "Missing user reference", ~has_value(frame.user_id), "Inspect source key")
        record(table, "User ID missing from raw users", has_value(frame.user_id) & ~frame.user_id.isin(known_ids), "True orphan candidate")
        record(table, "Rows linked to reviewed QA IDs", frame.user_id.isin(qa_ids), "Raw count includes duplicate copies")

record("users", "Reviewed QA names", qa_mask, "Candidate for shared QA exclusion")
names = raw_users.full_name.str.strip().str.casefold()
record("users", "Other possible test names", names.str.contains(r"\b(?:qa|test|demo|fake)\b|تجرب", na=False) & ~qa_mask, "Manual review only")
source = raw_users.utm_source.str.strip().str.casefold()
record("users", "Blank UTM source", ~has_value(raw_users.utm_source), "Preserve missing attribution")
record("users", "Nonblank source outside known aliases", has_value(raw_users.utm_source) & ~source.isin(KNOWN_SOURCES), "Review mapping")
record("users", "KYC timestamp before signup", dates(raw_users.kyc_verified_ts).lt(dates(raw_users.signup_ts)), "Inspect chronology")

txn = tables["transactions"]
record("transactions", "Transaction before signup", dates(txn.txn_ts).lt(txn.user_id.map(signups)), "Inspect chronology")
record("transactions", "Unexpected or missing status", ~txn.status.isin(["success", "failed"]), "README domain")
record("transactions", "Unexpected or missing transaction type", ~txn.txn_type.isin(
    ["deposit", "card_purchase", "p2p_send", "withdrawal", "bill_payment"]
), "README domain")
record("transactions", "Nonpositive deposit amount", txn.txn_type.eq("deposit") & numbers(txn.amount_egp).le(0), "Inspect source amount")

loans = tables["loans"]
term = dates(loans.due_date) - dates(loans.disbursed_date)
record("loans", "Loan term differs from 14 days", term.notna() & term.ne(pd.Timedelta(days=14)), "README term")
record("loans", "Last repayment before disbursal", dates(loans.last_repayment_date).lt(dates(loans.disbursed_date)), "Inspect chronology")
principal, fee, repaid = map(numbers, [loans.principal_egp, loans.fee_egp, loans.amount_repaid_egp])
record("loans", "Fee differs from 8% of principal", (fee - .08 * principal).abs().gt(.01), "README fee")
record("loans", "Repayment above principal plus fee", repaid.gt(principal + fee + .01), "Review amount")

plus = tables["plus_subscriptions"]
record("plus_subscriptions", "End before start", dates(plus.plus_end_date).lt(dates(plus.plus_start_date)), "Inspect chronology")
record("plus_subscriptions", "Monthly fee differs from 49 EGP", numbers(plus.monthly_fee_egp).sub(49).abs().gt(.01), "README fee")

cvm = tables["cvm_campaigns"]
record("cvm_campaigns", "Unexpected or missing group", ~cvm["group"].isin(["treatment", "holdout"]), "README domain")
record("cvm_campaigns", "Treatment missing send time", cvm["group"].eq("treatment") & ~has_value(cvm.sent_ts), "Review messaging evidence")
record("cvm_campaigns", "Holdout with send timestamp", cvm["group"].eq("holdout") & has_value(cvm.sent_ts), "Check whether field represents campaign time or actual send")

marketing = tables["marketing_spend"]
record("marketing_spend", "Unexpected or missing channel", ~marketing.channel.isin(CHANNELS), "Review channel domain")
record("marketing_spend", "Clicks above impressions", numbers(marketing.clicks).gt(numbers(marketing.impressions)), "Inspect platform definitions")
observed = pd.MultiIndex.from_arrays([dates(marketing.date).dt.normalize(), marketing.channel])
expected = pd.MultiIndex.from_product([Q3_DATES, CHANNELS], names=["date", "channel"])
coverage_missing = expected.difference(observed)
coverage_unexpected = observed.difference(expected)

quality = pd.DataFrame(quality_rows)
print("Checks with matching rows")
display(quality.loc[quality.rows.gt(0)].reset_index(drop=True))
print(f"Marketing date/channel coverage: {len(coverage_missing)} missing pairs, "
      f"{len(coverage_unexpected)} unexpected pairs")
if len(coverage_missing):
    display(coverage_missing.to_frame(index=False))
if len(coverage_unexpected):
    display(coverage_unexpected.to_frame(index=False))

# Reconcile overlapping QA and duplicate counts without modifying the source.
txn_once = txn.drop_duplicates()
transaction_audit = pd.Series({
    "raw_rows": len(txn), "exact_duplicate_copies": int(txn.duplicated().sum()),
    "rows_after_exact_deduplication": len(txn_once), "reviewed_qa_users": int(qa_mask.sum()),
    "raw_qa_transaction_rows": int(txn.user_id.isin(qa_ids).sum()),
    "distinct_qa_transaction_rows": int(txn_once.user_id.isin(qa_ids).sum()),
    "distinct_nonblank_true_orphan_rows": int((has_value(txn_once.user_id) & ~txn_once.user_id.isin(known_ids)).sum()),
}, name="count")
display(transaction_audit.to_frame())

display_details("All rule counts, including zero matches", [("Rules", quality)])
for (table, check), sample in examples.items():
    display_details(f"{table} | {check}", [("First five matching raw rows", sample)])

Checks with matching rows


,table,check,rows,pct_of_table,interpretation
0,transactions,Exact duplicate copies beyond first,577,1.36,Candidate for exact deduplication
1,transactions,Repeated complete key rows,1154,2.72,Includes original and repeated rows
2,transactions,Rows linked to reviewed QA IDs,689,1.63,Raw count includes duplicate copies
3,users,Reviewed QA names,30,0.48,Candidate for shared QA exclusion
4,users,Blank UTM source,794,12.74,Preserve missing attribution


Marketing date/channel coverage: 0 missing pairs, 0 unexpected pairs


,count
raw_rows,42378
exact_duplicate_copies,577
rows_after_exact_deduplication,41801
reviewed_qa_users,30
raw_qa_transaction_rows,689
distinct_qa_transaction_rows,682
distinct_nonblank_true_orphan_rows,0


,table,check,rows,pct_of_table,interpretation
0,users,Exact duplicate copies beyond first,0,0.00,Candidate for exact deduplication
1,users,Missing key component,0,0.00,Review intended grain
2,users,Repeated complete key rows,0,0.00,Includes original and repeated rows
3,users,Distinct rows sharing key,0,0.00,Inspect conflicting fields before choosing a rule
4,users,signup_ts: date parse failure,0,0.00,Inspect source
5,users,signup_ts: after extract day,0,0.00,Review timestamp
6,users,kyc_verified_ts: date parse failure,0,0.00,Inspect source
7,users,kyc_verified_ts: after extract day,0,0.00,Review timestamp
8,transactions,Exact duplicate copies beyond first,577,1.36,Candidate for exact deduplication
9,transactions,Missing key component,0,0.00,Review intended grain


,txn_id,user_id,txn_ts,txn_type,amount_egp,status
83,5009686,101033,2026-07-06 00:30:55,deposit,1000,success
277,5003329,100332,2026-07-09 17:26:48,deposit,200,success
634,5009130,100986,2026-07-14 02:50:52,deposit,500,success
727,5005043,100541,2026-07-15 02:49:00,deposit,200,success
899,5007439,100784,2026-07-16 16:54:45,deposit,1000,success


,txn_id,user_id,txn_ts,txn_type,amount_egp,status
82,5009686,101033,2026-07-06 00:30:55,deposit,1000,success
83,5009686,101033,2026-07-06 00:30:55,deposit,1000,success
276,5003329,100332,2026-07-09 17:26:48,deposit,200,success
277,5003329,100332,2026-07-09 17:26:48,deposit,200,success
633,5009130,100986,2026-07-14 02:50:52,deposit,500,success


,txn_id,user_id,txn_ts,txn_type,amount_egp,status
0,5041624,106221,2026-07-01 02:56:43,deposit,100000,success
1,5041625,106221,2026-07-01 05:56:43,deposit,20000,success
2,5041626,106221,2026-07-01 08:56:43,deposit,100000,success
3,5041627,106221,2026-07-01 11:56:43,deposit,100000,success
5,5041628,106221,2026-07-01 14:56:43,deposit,50000,success


,user_id,signup_ts,full_name,governorate,utm_source,utm_campaign,kyc_verified_ts
649,106224,2026-09-21 05:28:52,test qa,القاهرة,<NA>,<NA>,2026-09-21 05:33:52
688,106228,2026-08-25 06:20:21,test qa,القاهرة,<NA>,<NA>,2026-08-25 06:25:21
713,106230,2026-08-17 10:41:52,test test,القاهرة,<NA>,<NA>,2026-08-17 10:46:52
777,106221,2026-07-01 02:56:43,Test User,القاهرة,<NA>,<NA>,2026-07-01 03:01:43
796,106207,2026-08-01 15:22:50,test qa,القاهرة,<NA>,<NA>,2026-08-01 15:27:50


,user_id,signup_ts,full_name,governorate,utm_source,utm_campaign,kyc_verified_ts
0,101286,2026-07-30 04:52:12,كريم قناوي,القليوبية,<NA>,<NA>,2026-07-31 01:41:49
22,102864,2026-08-17 18:04:28,شهد حسن,الجيزة,<NA>,<NA>,2026-08-20 02:39:32
23,105934,2026-09-17 10:37:50,محمد فؤاد,الإسكندرية,<NA>,<NA>,2026-09-18 00:48:09
36,101861,2026-08-27 11:08:53,عمر إبراهيم,الجيزة,<NA>,<NA>,<NA>
42,103655,2026-08-05 13:27:19,محمود فتحي,القاهرة,<NA>,<NA>,<NA>


## Per-table column profiles

Repeated-value counts exclude nulls. An extra occurrence is a row beyond the first occurrence of a value. Repetition is expected in many columns. ID continuity describes the observed range and does not imply that skipped IDs are missing records. Date checks use UTC and allow any timestamp on extract day. Numeric buckets describe the distribution and the extremes are candidates for review.

In [4]:
def profile_table(table):
    frame = tables[table]
    spec = SCHEMA[table]
    print(f"{table}: {len(frame):,} rows, {len(frame.columns)} columns")
    display(column_summary(frame, spec))
    for column, series in frame.items():
        role = role_of(column, spec)
        display_details(f"{column} ({role})", column_details(series, role))

### users

In [5]:
profile_table("users")

users: 6,230 rows, 7 columns


,role,rows,missing,missing_pct,whitespace_only,distinct_non_null,distinct_values_repeated,extra_occurrences,extra_occurrences_pct
column,,,,,,,,,
user_id,ID,6230,0,0.00,0,6230,0,0,0.00
signup_ts,date,6230,0,0.00,0,6228,2,2,0.03
full_name,text,6230,0,0.00,0,425,343,5805,93.18
governorate,text,6230,0,0.00,0,10,10,6220,99.84
utm_source,text,6230,794,12.74,0,20,20,5416,86.93
utm_campaign,text,6230,1309,21.01,0,7,7,4914,78.88
kyc_verified_ts,date,6230,2470,39.65,0,3759,1,1,0.02


,value
earliest,2026-07-01 00:22:55+00:00
latest,2026-09-30 23:47:10+00:00
parse_failures,0
after_extract_day,0
,rows
signup_ts,
2026-07-01,56
2026-07-02,61
2026-07-03,64
2026-07-04,61


,rows
full_name,
أحمد علي,39
عمرو قناوي,36
آية حسن,35
شهد سامي,35
شريف عماد,35
شهد فتحي,34
إيمان سامي,33
منى جمال,33
هدى حسن,33


,rows
governorate,
القاهرة,1900
الجيزة,1367
الإسكندرية,895
الشرقية,390
الدقهلية,375
القليوبية,371
سوهاج,245
المنيا,242
الغربية,239


,rows
utm_source,
<NA>,794
google,576
meta,323
fb,312
Facebook Ads,309
facebook,308
adwords,293
instagram,293
ig,288


,rows
utm_campaign,
<NA>,1309
invite_friend,741
uac_q3,734
meta_retarget,724
meta_install_q3,711
search_brand,696
meta_lal_aug,665
tt_spark_q3,650
,rows


,value
earliest,2026-07-01 03:01:43+00:00
latest,2026-10-05 10:28:50+00:00
parse_failures,0
after_extract_day,0
,rows
kyc_verified_ts,
2026-07-01,6
2026-07-02,19
2026-07-03,35
2026-07-04,28


### transactions

In [5]:
profile_table("transactions")

transactions: 42,378 rows, 6 columns


,role,rows,missing,missing_pct,whitespace_only,distinct_non_null,distinct_values_repeated,extra_occurrences,extra_occurrences_pct
column,,,,,,,,,
txn_id,ID,42378,0,0.0,0,41801,577,577,1.36
user_id,ID,42378,0,0.0,0,3668,2447,38710,91.34
txn_ts,date,42378,0,0.0,0,41714,662,664,1.57
txn_type,text,42378,0,0.0,0,5,5,42373,99.99
amount_egp,numeric,42378,0,0.0,0,22665,5449,19713,46.52
status,text,42378,0,0.0,0,2,2,42376,100.00


,value
earliest,2026-07-01 02:56:43+00:00
latest,2026-11-04 23:56:48+00:00
parse_failures,0
after_extract_day,0
,rows
txn_ts,
2026-07-01,10
2026-07-02,6
2026-07-03,18
2026-07-04,15


,rows
txn_type,
card_purchase,16098
deposit,12708
p2p_send,6030
bill_payment,3776
withdrawal,3766
,rows
txn_type,
card_purchase,16098
deposit,12708


,value
count,42378.0
mean,1785.771983
std,12276.194444
min,7.82
1%,40.2277
5%,71.304
25%,158.6525
50%,278.23
75%,500.0
95%,5000.0


,rows
status,
success,39059
failed,3319
,rows
status,
success,39059
failed,3319


### loans

In [6]:
profile_table("loans")

loans: 1,310 rows, 8 columns


,role,rows,missing,missing_pct,whitespace_only,distinct_non_null,distinct_values_repeated,extra_occurrences,extra_occurrences_pct
column,,,,,,,,,
loan_id,ID,1310,0,0.00,0,1310,0,0,0.00
user_id,ID,1310,0,0.00,0,1310,0,0,0.00
disbursed_date,date,1310,0,0.00,0,87,86,1223,93.36
principal_egp,numeric,1310,0,0.00,0,5,5,1305,99.62
fee_egp,numeric,1310,0,0.00,0,5,5,1305,99.62
due_date,date,1310,0,0.00,0,87,86,1223,93.36
amount_repaid_egp,numeric,1310,0,0.00,0,72,6,1238,94.50
last_repayment_date,date,1310,49,3.74,0,116,107,1145,87.40


,value
earliest,2026-07-05 00:00:00+00:00
latest,2026-09-29 00:00:00+00:00
parse_failures,0
after_extract_day,0
,rows
disbursed_date,
2026-07-05,2
2026-07-06,1
2026-07-07,4
2026-07-08,3


,value
count,1310.0
mean,1309.541985
std,749.958351
min,500.0
1%,500.0
5%,500.0
25%,500.0
50%,1000.0
75%,1500.0
95%,3000.0


,value
count,1310.0
mean,104.763359
std,59.996668
min,40.0
1%,40.0
5%,40.0
25%,40.0
50%,80.0
75%,120.0
95%,240.0


,value
earliest,2026-07-19 00:00:00+00:00
latest,2026-10-13 00:00:00+00:00
parse_failures,0
after_extract_day,0
,rows
due_date,
2026-07-19,2
2026-07-20,1
2026-07-21,4
2026-07-22,3


,value
count,1310.0
mean,1294.282634
std,859.376451
min,0.0
1%,0.0
5%,89.568
25%,540.0
50%,1080.0
75%,1620.0
95%,3240.0


,value
earliest,2026-07-12 00:00:00+00:00
latest,2026-11-05 00:00:00+00:00
parse_failures,0
after_extract_day,0
,rows
last_repayment_date,
2026-07-12,1
2026-07-13,1
2026-07-14,3
2026-07-16,2


### plus_subscriptions

In [7]:
profile_table("plus_subscriptions")

plus_subscriptions: 111 rows, 4 columns


,role,rows,missing,missing_pct,whitespace_only,distinct_non_null,distinct_values_repeated,extra_occurrences,extra_occurrences_pct
column,,,,,,,,,
user_id,ID,111,0,0.00,0,111,0,0,0.00
plus_start_date,date,111,0,0.00,0,49,30,62,55.86
plus_end_date,date,111,82,73.87,0,22,6,7,6.31
monthly_fee_egp,numeric,111,0,0.00,0,1,1,110,99.10


,value
earliest,2026-08-14 00:00:00+00:00
latest,2026-11-02 00:00:00+00:00
parse_failures,0
after_extract_day,0
,rows
plus_start_date,
2026-08-14,1
2026-08-15,2
2026-08-16,4
2026-08-18,3


,value
earliest,2026-09-15 00:00:00+00:00
latest,2026-11-04 00:00:00+00:00
parse_failures,0
after_extract_day,0
,rows
plus_end_date,
2026-09-15,1
2026-09-17,2
2026-09-19,1
2026-09-22,2


,value
count,111.0
mean,49.0
std,0.0
min,49.0
1%,49.0
5%,49.0
25%,49.0
50%,49.0
75%,49.0
95%,49.0


### cvm_campaigns

In [8]:
profile_table("cvm_campaigns")

cvm_campaigns: 697 rows, 4 columns


,role,rows,missing,missing_pct,whitespace_only,distinct_non_null,distinct_values_repeated,extra_occurrences,extra_occurrences_pct
column,,,,,,,,,
user_id,ID,697,0,0.00,0,697,0,0,0.00
campaign_id,ID,697,0,0.00,0,1,1,696,99.86
group,text,697,0,0.00,0,2,2,695,99.71
sent_ts,date,697,137,19.66,0,216,164,344,49.35


,rows
campaign_id,
plus_push_aug,697
,rows
campaign_id,
plus_push_aug,697


,rows
group,
treatment,560
holdout,137
,rows
group,
treatment,560
holdout,137


,value
earliest,2026-08-15 10:00:00+00:00
latest,2026-08-15 13:59:00+00:00
parse_failures,0
after_extract_day,0
,rows
sent_ts,
2026-08-15,560


### marketing_spend

In [9]:
profile_table("marketing_spend")

marketing_spend: 276 rows, 6 columns


,role,rows,missing,missing_pct,whitespace_only,distinct_non_null,distinct_values_repeated,extra_occurrences,extra_occurrences_pct
column,,,,,,,,,
date,date,276,0,0.0,0,92,92,184,66.67
channel,text,276,0,0.0,0,3,3,273,98.91
spend_egp,numeric,276,0,0.0,0,276,0,0,0.00
impressions,numeric,276,0,0.0,0,174,70,102,36.96
clicks,numeric,276,0,0.0,0,119,66,157,56.88
platform_reported_installs,numeric,276,0,0.0,0,62,39,214,77.54


,value
earliest,2026-07-01 00:00:00+00:00
latest,2026-09-30 00:00:00+00:00
parse_failures,0
after_extract_day,0
,rows
date,
2026-07-01,3
2026-07-02,3
2026-07-03,3
2026-07-04,3


,rows
channel,
Meta,92
Google Ads,92
TikTok,92
,rows
channel,
meta,92
google ads,92
tiktok,92


,value
count,276.0
mean,1291.80692
std,803.287866
min,56.02
1%,71.2625
5%,226.1775
25%,556.4775
50%,1203.185
75%,1858.255
95%,2756.5


,value
count,276.0
mean,5564.583333
std,4530.197371
min,625.0
1%,786.75
5%,1199.0
25%,1882.5
50%,4250.0
75%,8000.0
95%,15681.5


,value
count,276.0
mean,70.655797
std,44.090168
min,5.0
1%,6.75
5%,18.5
25%,42.0
50%,59.0
75%,90.0
95%,172.5


,value
count,276.0
mean,22.981884
std,14.747685
min,2.0
1%,2.0
5%,7.0
25%,13.0
50%,19.5
75%,26.25
95%,56.25
